# Structured extraction with an LLM

Turns the raw *California Culinary Map* text into validated JSON. A local LLM (`llama3` via Ollama) extracts one record per restaurant, a Pydantic schema validates it, and a self-repair loop asks the model to fix invalid output (up to 5 retries). Output: `data/structured_restaurant_data.json`.

> Part of **Connoisseur Companion**. Adapted from the IBM *RAG and Agentic AI* course on Coursera, reworked to run fully locally with Ollama.

In [ ]:
# Run from the repo: notebooks read and write files in `data/`
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent / "data")
print("Working directory:", Path.cwd())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import os

import ollama

# Libraries and codes to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

## Load the data and display the texts

In [ ]:
### 1.1: Define the file_path to the text file
file_path =  "California-Culinary-Map.txt"

### 1.2: Open the text file
with open(file_path, 'r') as file:
    data = file.read()

### 1.3: Print the first 100 characters of the restaurant data
print(data[:100])

In [ ]:
### 2.1: Split the restaurant paragraphs into list (hint: use .split('\n\n')
restaurant_list = data.split('\n\n') 

### 2.2: Since the first item is the dataset name, we remove it
restaurant_list = restaurant_list[1:]

### 2.3: Print out the number of restaurants we have (hint: use the len() function)
print(len(restaurant_list))

### 2.4: Print out the first item to have a closer look at the content
print(restaurant_list[0])

## Define the LLM

### Implement the LLM function

In [ ]:
def llm_model(system_msg, prompt_txt):
    #system_msg: the system message given to the LLM
    #prompt_txt: the user prompt
    
    model = "llama3"

    ### 1.2: Define the messages
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt_txt}
    ]

    ### 1.3: Get the final response output and return it
    response = ollama.chat(model=model, messages=messages)
    output_text = response["message"]["content"]
    
    return output_text

### Test your llm_model()

In [ ]:
import time

def safe_llm_call(system_msg, prompt_txt, retries=3):
    for i in range(retries):
        try:
            return llm_model(system_msg, prompt_txt)
        except Exception:
            time.sleep(2)
    return "Failed after retries"

In [ ]:
system_msg = "You are a helpful assistant."
prompt_txt = "Which place is warmer in winter? Hawaii or Greenland?"
print(safe_llm_call(system_msg, prompt_txt))

## Prompt Engineering

### Define the template

In [ ]:
EXAMPLE_RESTAURANT_PARAGRAPH = restaurant_list[1] #use the second restaurant paragraph as the example
EXAMPLE_OUTPUT = """
    {{
    "name": "Mar de Cortez",
    "location": "Santa Monica",
    "type": "casual taqueria",
    "food_style": "Baja-style seafood",
    "rating": 4.2,
    "price_range": 1,
    "signatures": [
        "beer-battered snapper tacos",
        "zesty octopus ceviche"
    ],
    "vibe": "salt-air energy",
    "environment": "a premier sun-drenched spot for open-air dining near the pier."
    "shortcomings": []
    }}
"""

### Design your prompt here
def restaurant_data_structure_prompt_generation(restaurant_paragraph):
    base_system_msg = f"""
    Respond with ONLY the JSON object. Do not include any explanation, preamble, or markdown formatting. Your entire response must be valid JSON starting with '{{' and ending with '}}'.
    """
    
    base_user_prompt = f"""
    Task:
    Your task is to extract structured JSON data from the given restaurant description. the JSON schema shuold include the following fields in the format that 
    is shown below:

    Restaurant description:
    {restaurant_paragraph}

    Example:
    Input Restaurant Description: {EXAMPLE_RESTAURANT_PARAGRAPH}
    Output:
    {EXAMPLE_OUTPUT}
    
    """
    return base_system_msg, base_user_prompt

### Test your prompts

In [ ]:
# Unit test:
restaurant_paragraph = restaurant_list[0]
base_system_msg, base_user_prompt = restaurant_data_structure_prompt_generation(restaurant_paragraph=restaurant_paragraph)

test_response = llm_model(system_msg=base_system_msg, prompt_txt=base_user_prompt)
print(test_response)

### Validate the LLM outputs

In [ ]:
# Validation
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional

### 3.1. Define the schema
class Restaurant(BaseModel):
    name: str
    location: str
    type: str
    food_style: str
    rating: Optional[float] = None
    price_range: Optional[int] = None
    signatures: List[str] = Field(default_factory=list)
    vibe: Optional[str] = None
    environment: str
    shortcomings: List[str] = Field(default_factory=list)


### 3.2. Use the validation method to validate the test_response from the unit test
try:
    restaurant_data = Restaurant.model_validate_json(test_response)
    print(f"Success! Validated: {restaurant_data.name}")
except ValidationError as e:
    print(f"Validation failed: {e.json()}")

## Structure all the restaurant data

### Define prompts to have an LLM that auto repairs outputs to JSON format

In [ ]:
def JSON_auto_repair_prompts(candidate_json_output, error_message):
    auto_repair_system_msg = """
    You are an expert JSON repair assistant. You excel at finding and fixing errors in JSON data.
    """
    auto_repair_prompt = f"""
    The following JSON data has errors:
    {candidate_json_output}

    Error message:
    {error_message}

    Please fix the errors and return the corrected JSON data with the exact and ONLY JSON schema.
    """
    return auto_repair_system_msg, auto_repair_prompt

In [ ]:
structured_restaurant_lists = []
max_retries = 0
for i, restaurant_paragraph in enumerate(restaurant_list):
    ### 2.1: Produce your initial output
    candidate_json_output = llm_model(*restaurant_data_structure_prompt_generation(restaurant_paragraph=restaurant_paragraph))
    ### 2.2: Validation and Auto Correction loop on the output 
    while max_retries < 5:
        try:
            restaurant_data = Restaurant.model_validate_json(candidate_json_output)
            break  # Exit the loop if validation is successful
        except ValidationError as e:
            error_message = e.json()
            auto_repair_system_msg, auto_repair_prompt = JSON_auto_repair_prompts(candidate_json_output, error_message)
            candidate_json_output = llm_model(auto_repair_system_msg, auto_repair_prompt)
            max_retries += 1
   

    ### 2.3: Append your finalized response to the structured_restaurant_lists
    structured_restaurant_lists.append(candidate_json_output)

    # A manual progress bar
    if (i+1)%20 == 0:
        print(f'{i+1} out of {len(restaurant_list)} is done')

# A final message to notify the completion
print('ALL DONE!!')

### Save the list to a JSON file

In [ ]:
### Print the 50th item in the structured_restaurant_lists
print(structured_restaurant_lists[49])

In [ ]:
for i, r in enumerate(structured_restaurant_lists):
    try:
        json.loads(r)
    except json.JSONDecodeError as e:
        print(i, repr(r[:100]))

In [ ]:
structured_restaurant_lists_json = []
for i, response in enumerate(structured_restaurant_lists):
    start = response.find('{')
    end = response.rfind('}')
    if start == -1 or end == -1 or start > end:
        print(f"Skipping item {i}: no JSON object found in response")
        continue
    cleaned_response = response[start:end + 1]
    try:
        structured_restaurant_lists_json.append(json.loads(cleaned_response))
    except json.JSONDecodeError as e:
        print(f"Skipping item {i}: still invalid JSON after cleanup ({e})")

#For each item in the restaurant list, assign it with an itemId to be consistent with the one in the user review data:
for i, response in enumerate(structured_restaurant_lists_json):
    response['itemId'] = 1000001 + i
    structured_restaurant_lists_json[i] = response
    
filename = 'structured_restaurant_data.json'
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(structured_restaurant_lists_json, f, indent=4)